# Pipeline tái lập Triple XGBoost

Notebook này chỉ giữ đúng đường tái lập cần thiết:
1. đọc manifest đã có sẵn;
2. xem một mẫu feature sequence;
3. làm giàu theo thời gian lên 504 chiều;
4. tổng hợp tabular bằng `tsfresh`-like features;
5. chạy Triple XGBoost fusion trên cùng manifest.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engagement_daisee.common.config import FOUR_CLASS_FEATURE_MANIFEST_CSV
from engagement_daisee.common.manifest import normalize_manifest_columns
from engagement_daisee.ml.train import _build_feature_matrix, _sequence_to_tsfresh_like_features
from engagement_daisee.multiclass.train_triple_xgb import train_triple_xgb
from engagement_daisee.app.focus_monitor import LEFT_EYE, RIGHT_EYE, LEFT_IRIS, RIGHT_IRIS, NOSE_TIP, CHIN, UPPER_LIP, LOWER_LIP

ROOT = Path.cwd()
MANIFEST = ROOT / 'data/processed/feature_manifest.csv'
FIG_DIR = ROOT / 'checkpoints/reports/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

manifest = normalize_manifest_columns(pd.read_csv(MANIFEST, low_memory=False))
manifest['split'] = manifest['split'].astype(str).str.strip().str.lower()
manifest['label'] = manifest['label'].astype(int)
manifest.head(3)


## 1) Manifest và cấu hình mẫu

In [ ]:
sample = manifest.iloc[[0]].copy()
print('rows:', len(manifest))
print('columns:', list(manifest.columns))
print('sample video_id:', sample.iloc[0].get('video_id'))
print('sample feature_path:', sample.iloc[0]['feature_path'])
print('sample label:', int(sample.iloc[0]['label']))
print('sample split:', sample.iloc[0]['split'])


## 2) Một sequence 168 chiều trên mỗi frame

In [ ]:
feature_path = Path(sample.iloc[0]['feature_path'])
sequence = np.load(feature_path)
print('sequence shape:', sequence.shape)
print('dtype:', sequence.dtype)
print('first frame, first 8 dims:', np.round(sequence[0, :8], 4))


In [ ]:
# Plot a few dimensions so the temporal evolution is visible.
fig, ax = plt.subplots(figsize=(10, 4))
for idx, color in zip([0, 1, 2, 3, 4], ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']):
    ax.plot(sequence[:, idx], label=f'feat[{idx}]', color=color, linewidth=1.5)
ax.set_title('Mẫu biến thiên của vài chiều feature theo thời gian')
ax.set_xlabel('frame')
ax.set_ylabel('value')
ax.legend(ncol=5, fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig_path = FIG_DIR / '01_sequence_sample.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
fig


## 3) Làm giàu theo thời gian lên 504 chiều

In [ ]:
def temporal_enrich(seq: np.ndarray) -> np.ndarray:
    seq = np.asarray(seq, dtype=np.float32)
    if seq.ndim != 2:
        raise ValueError(f'Expected 2D sequence, got {seq.shape}')
    diff = np.diff(seq, axis=0)
    if diff.size == 0:
        diff = np.zeros_like(seq)
    std = seq.std(axis=0, keepdims=True)
    std = np.repeat(std, repeats=seq.shape[0], axis=0)
    return np.concatenate([seq, np.vstack([np.zeros((1, seq.shape[1]), dtype=np.float32), diff]), std], axis=1)

enriched = temporal_enrich(sequence)
print('enriched shape:', enriched.shape)
print('expected frame dim:', sequence.shape[1] * 3)
print('first frame enriched head:', np.round(enriched[0, :12], 4))


## 4) Từ sequence sang vector tabular cho XGBoost

In [ ]:
tsfresh_like = _sequence_to_tsfresh_like_features(enriched)
print('tabular feature length:', len(tsfresh_like))
print('first 15 values:', np.round(tsfresh_like[:15], 4))


## 5) Landmark groups dùng trong MediaPipe

In [ ]:
landmark_groups = {
    'LEFT_EYE': LEFT_EYE,
    'RIGHT_EYE': RIGHT_EYE,
    'LEFT_IRIS': LEFT_IRIS,
    'RIGHT_IRIS': RIGHT_IRIS,
    'NOSE_TIP': [NOSE_TIP],
    'CHIN': [CHIN],
    'UPPER_LIP': [UPPER_LIP],
    'LOWER_LIP': [LOWER_LIP],
}
for name, idxs in landmark_groups.items():
    print(f'{name:>10}: {idxs}')


## 6) Chạy Triple XGBoost fusion trên manifest có sẵn

In [ ]:
output_dir = ROOT / 'checkpoints/runs/product_4class_fixed_triple_xgb'
summary = train_triple_xgb(type('Args', (), {
    'manifest': MANIFEST,
    'output_dir': output_dir,
    'feature_mode': 'tsfresh',
    'latency_warmup': 30,
    'latency_iters': 200,
})())
print(json.dumps({
    'run_root': summary['run_root'],
    'fusion_accuracy': summary['fusion_report']['test_metrics']['accuracy'],
    'fusion_balanced_accuracy': summary['fusion_report']['test_metrics']['balanced_accuracy'],
    'fusion_f1_macro': summary['fusion_report']['test_metrics']['f1_macro'],
    'fusion_latency_ms': summary['fusion_report']['latency']['model_side']['latency_ms_mean'],
}, indent=2))


## 7) Tổng kết

In [ ]:
summary = {
    'manifest_rows': len(manifest),
    'sequence_shape': tuple(sequence.shape),
    'enriched_shape': tuple(enriched.shape),
    'tabular_dim': int(len(tsfresh_like)),
    'figure_path': str(fig_path),
    'summary_json': str(output_json),
}
summary
